# Credit Card Default Prediction
**Dataset:** UCI Credit Card Default (30.000 clienti, Taiwan 2005)

**Obiettivo:** Predire la probabilità di default nel mese successivo sulla base di dati demografici, limiti di credito e storico pagamenti.

**Pipeline:** EDA → Feature Engineering → Preprocessing → Logistic Regression → Random Forest → Confronto modelli

**Metrica principale:** ROC-AUC (standard in credit risk)

## 1. Import e caricamento dati

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print('Librerie importate correttamente')

In [ ]:
url = "C:/Users/Utente/Desktop/Python Carlo/Input/default of credit card clients.xls"
df = pd.read_excel(url, header=1)  # header=1: la riga 0 è un titolo, i nomi colonne sono in riga 1
df.head()

## 2. Exploratory Data Analysis (EDA)

Prima di modellare esploriamo la struttura del dataset: dimensioni, tipi di dato, valori mancanti, duplicati e bilanciamento del target.

In [ ]:
print('Shape:', df.shape)
print('\nTipi di dato:')
print(df.dtypes)
print('\nValori mancanti per colonna:')
print(df.isnull().sum())
print('\nDuplicati:', df.duplicated().sum())

In [ ]:
# Bilanciamento del target
print(df['default payment next month'].value_counts())
df['default payment next month'].value_counts(normalize=True).round(2)

**Osservazione:** Il target è sbilanciato — 78% non-default vs 22% default. 
Questo influenza la scelta della metrica (AUC invece di accuracy) e dei parametri del modello (class_weight).

In [ ]:
df.describe()

### Dizionario variabili

- **LIMIT_BAL** — limite di credito in NT dollar
- **SEX** — 1=maschio, 2=femmina
- **EDUCATION** — 1=graduate school, 2=university, 3=high school, 4=other (valori 0/5/6 = anomali, accorpati a 4)
- **MARRIAGE** — 1=married, 2=single, 3=other
- **AGE** — età in anni
- **PAY_0...PAY_6** — storico pagamenti sett→apr: -1=regolare, 0=revolving, 1+=mesi ritardo
- **BILL_AMT1...BILL_AMT6** — importo estratto conto sett→apr
- **PAY_AMT1...PAY_AMT6** — importo pagato sett→apr

### Analisi correlazioni

In [ ]:
# Correlazione con il target
correlazioni_target = df.corr()['default payment next month'].sort_values(ascending=False)
print(correlazioni_target)

**Osservazione:** PAY_0 (0.32) è il predittore singolo più forte. 
Correlazioni globalmente basse — tipico del credit risk reale dove le banche già selezionano i clienti a monte.

In [ ]:
# Heatmap correlazioni
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Multicollinearità tra variabili interne
corr_matrix = df.corr()
high_corr = corr_matrix.abs().unstack().sort_values(ascending=False)
high_corr = high_corr[high_corr < 1]

# Rimuovo coppie duplicate (A,B) = (B,A)
high_corr_clean = high_corr[
    high_corr.index.get_level_values(0) < high_corr.index.get_level_values(1)
]
print('Coppie con correlazione > 0.7:')
print(high_corr_clean[high_corr_clean > 0.7])

**Osservazione:** I BILL_AMT (estratti conto) sono tutti correlati tra loro con valori 0.86-0.95. 
Si tratta di autocorrelazione tipica delle serie temporali — il saldo di un mese dipende da quello precedente. 
Problema principale per Logistic Regression (instabilità coefficienti), meno critico per Random Forest.

## 3. Feature Engineering

Invece di eliminare le variabili correlate, creiamo feature più informative che catturano la dinamica temporale del comportamento di pagamento.

In [ ]:
# Trend del saldo: differenza media mese-su-mese (sett vs apr)
df['bill_trend_overall'] = df['BILL_AMT1'] - df['BILL_AMT6']
# Positivo = debito crescente nel tempo (segnale di rischio)

# Pay ratio: quanto paga rispetto al dovuto (mese più recente)
# abs() evita divisione per zero quando BILL_AMT1 = -1
df['pay_ratio'] = df['PAY_AMT1'] / (df['BILL_AMT1'].abs() + 1)

# Capacità di pagamento storica: totale pagato negli ultimi 6 mesi
df['total_paid'] = df[['PAY_AMT1','PAY_AMT2','PAY_AMT3',
                        'PAY_AMT4','PAY_AMT5','PAY_AMT6']].sum(axis=1)

# Persistenza ritardi: somma dei mesi in ritardo
df['total_delay'] = df[['PAY_0','PAY_2','PAY_3',
                         'PAY_4','PAY_5','PAY_6']].sum(axis=1)

print('Feature ingegnerizzate:', ['bill_trend_overall', 'pay_ratio', 'total_paid', 'total_delay'])

Le 4 feature catturano angolazioni diverse del comportamento del cliente:

- **bill_trend_overall** → dinamica del debito nel tempo
- **pay_ratio** → comportamento di pagamento recente
- **total_paid** → capacità di pagamento storica
- **total_delay** → persistenza dei ritardi

## 4. Preprocessing

Tre step in ordine: encoding variabili categoriche → train/test split → scaling variabili numeriche.

Lo scaling viene eseguito **dopo** lo split per evitare data leakage (lo scaler impara media e std solo dal training set).

In [ ]:
# Fix valori anomali in EDUCATION (0, 5, 6 non documentati → accorpati a 4 = other)
df['EDUCATION'] = df['EDUCATION'].replace({5: 4, 6: 4})

# One-hot encoding variabili categoriche
# drop_first=True evita la dummy trap (multicollinearità perfetta)
df = pd.get_dummies(df, columns=['SEX', 'EDUCATION', 'MARRIAGE'], drop_first=True)
print('Colonne dopo encoding:', df.shape[1])

In [ ]:
# Train/test split 80/20
X = df.drop(columns=['default payment next month', 'ID'])
y = df['default payment next month']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

In [ ]:
# Scaling solo colonne numeriche continue (le dummy sono già 0/1)
colonne_numeriche = []
for col in X_train.select_dtypes(include='number').columns:
    if 'SEX' not in col and 'EDUCATION' not in col and 'MARRIAGE' not in col:
        colonne_numeriche.append(col)

scaler = StandardScaler()
X_train[colonne_numeriche] = scaler.fit_transform(X_train[colonne_numeriche])
X_test[colonne_numeriche] = scaler.transform(X_test[colonne_numeriche])
print('Scaling completato su', len(colonne_numeriche), 'colonne')

## 5. Modellazione

Confrontiamo quattro configurazioni:
1. Logistic Regression (baseline)
2. Logistic Regression con class_weight='balanced'
3. Random Forest (default)
4. Random Forest ottimizzato con GridSearchCV

**Metrica principale:** ROC-AUC — standard in credit risk, non sensibile allo sbilanciamento del target.

In [ ]:
# 1. Logistic Regression default
logistic = LogisticRegression()
logistic.fit(X_train, y_train)
y_pred = logistic.predict(X_test)
y_prob = logistic.predict_proba(X_test)[:, 1]

print('--- Logistic Regression default ---')
print(classification_report(y_test, y_pred))
print('AUC:', round(roc_auc_score(y_test, y_prob), 3))

In [ ]:
# 2. Logistic Regression con class_weight='balanced'
# Pesa gli errori sulla classe minoritaria proporzionalmente alla sua frequenza
log_bal = LogisticRegression(class_weight='balanced')
log_bal.fit(X_train, y_train)
y_pred_2 = log_bal.predict(X_test)
y_prob_2 = log_bal.predict_proba(X_test)[:, 1]

print('--- Logistic Regression balanced ---')
print(classification_report(y_test, y_pred_2))
print('AUC:', round(roc_auc_score(y_test, y_prob_2), 3))

In [ ]:
# 3. Random Forest default
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('--- Random Forest default ---')
print(classification_report(y_test, y_pred_rf))
print('AUC:', round(roc_auc_score(y_test, y_prob_rf), 3))

In [ ]:
# 4. Random Forest ottimizzato con GridSearchCV
# Ottimizza su roc_auc con 5-fold cross-validation
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'class_weight': ['balanced', None]
}

grid = GridSearchCV(rf, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print('Migliori parametri:', grid.best_params_)
print('Migliore AUC CV:', round(grid.best_score_, 3))

In [ ]:
# Modello finale con parametri ottimizzati
rf_best = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight=None, random_state=42)
rf_best.fit(X_train, y_train)
y_pred_rfbest = rf_best.predict(X_test)
y_prob_rfbest = rf_best.predict_proba(X_test)[:, 1]

print('--- Random Forest ottimizzato ---')
print(classification_report(y_test, y_pred_rfbest))
print('AUC:', round(roc_auc_score(y_test, y_prob_rfbest), 3))

### Check overfitting

Confronto performance train vs test per verificare che il modello non abbia memorizzato il training set.

In [ ]:
print('--- Performance TRAIN ---')
print(classification_report(y_train, rf_best.predict(X_train)))
print('--- Performance TEST ---')
print(classification_report(y_test, y_pred_rfbest))

## 6. Confronto modelli e conclusioni

In [ ]:
# Tabella di confronto finale
risultati = []
modelli = {
    'LogReg default': (y_pred, y_prob),
    'LogReg balanced': (y_pred_2, y_prob_2),
    'RF default': (y_pred_rf, y_prob_rf),
    'RF ottimizzato': (y_pred_rfbest, y_prob_rfbest)
}

for nome, (pred, prob) in modelli.items():
    report = classification_report(y_test, pred, output_dict=True)
    risultati.append({
        'Modello': nome,
        'AUC': round(roc_auc_score(y_test, prob), 3),
        'Recall classe 1': round(report['1']['recall'], 3),
        'Precision classe 1': round(report['1']['precision'], 3),
        'Accuracy': round(report['accuracy'], 3)
    })

df_risultati = pd.DataFrame(risultati)
print(df_risultati.to_string(index=False))

In [ ]:
# Feature importance del modello migliore
feat_imp = pd.Series(rf_best.feature_importances_, index=X_test.columns)
feat_imp.sort_values(ascending=False).plot(kind='bar', figsize=(14, 6))
plt.title('Feature Importance - Random Forest ottimizzato')
plt.tight_layout()
plt.show()

## Conclusioni

- **Random Forest ottimizzato** è il modello migliore con AUC 0.780
- **PAY_0 e total_delay** sono i predittori più importanti — il comportamento di pagamento recente e la storia di ritardi accumulata dominano sul profilo demografico
- Le variabili demografiche (SEX, EDUCATION, MARRIAGE) contribuiscono marginalmente
- Il **feature engineering** (total_delay, bill_trend_overall) ha aggiunto valore misurabile rispetto alle variabili grezze
- In un contesto bancario reale si abbasserebbe la soglia di classificazione per aumentare la recall sulla classe 1 (default), accettando più falsi positivi per minimizzare le perdite su crediti non recuperati